In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np

# Dataset limpo da fase 2 (sem placeholders)
df = pd.read_parquet("../data/processed/licitacoes_limpo.parquet")
print(df.shape)
df.columns.tolist()

(84262, 18)


['Número Licitação',
 'Código UG',
 'Nome UG',
 'Código Modalidade Compra',
 'Modalidade Compra',
 'Número Processo',
 'Objeto',
 'Situação Licitação',
 'Código Órgão Superior',
 'Nome Órgão Superior',
 'Código Órgão',
 'Nome Órgão',
 'UF',
 'Município',
 'Data Resultado Compra',
 'Data Abertura',
 'Valor Licitação',
 'arquivo_origem']

In [2]:
import sys
sys.path.append("..")
import pandas as pd
import numpy as np

df_lic = pd.read_parquet("../data/processed/licitacoes_limpo.parquet")
print("Licitações limpas:", df_lic.shape)

Licitações limpas: (84262, 18)


In [3]:
from src.carga import carregar_participantes

part = carregar_participantes("../data/raw")
print("Participantes:", part.shape, "| Fornecedores:", part["Código Participante"].nunique())

c:\auditoria-ml\notebooks\..\src\carga.py:30: DtypeWarning: Columns (0: Código Participante) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arq, sep=";", encoding="latin-1", decimal=",")
c:\auditoria-ml\notebooks\..\src\carga.py:30: DtypeWarning: Columns (0: Código Participante) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arq, sep=";", encoding="latin-1", decimal=",")
c:\auditoria-ml\notebooks\..\src\carga.py:30: DtypeWarning: Columns (0: Código Participante) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arq, sep=";", encoding="latin-1", decimal=",")
c:\auditoria-ml\notebooks\..\src\carga.py:30: DtypeWarning: Columns (0: Código Participante) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arq, sep=";", encoding="latin-1", decimal=",")
c:\auditoria-ml\notebooks\..\src\carga.py:30: DtypeWarning: Columns 

Participantes: (4607976, 14) | Fornecedores: 81766


In [4]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [5]:
from sklearn.ensemble import IsolationForest

# --- Feature engineering por fornecedor ---
g = part.groupby("Código Participante")
feat = pd.DataFrame({
    "n_licitacoes": g.size(),
    "n_vitorias": g["Flag Vencedor"].apply(lambda x: (x == "SIM").sum()),
    "n_ugs": g["Código UG"].nunique(),
    "n_modalidades": g["Modalidade Compra"].nunique(),
    "pct_dispensa": g["Modalidade Compra"].apply(lambda x: (x == "Dispensa de Licitação").mean()),
    "pct_inexig": g["Modalidade Compra"].apply(lambda x: (x == "Inexigibilidade de Licitação").mean()),
})
feat["taxa_vitoria"] = feat["n_vitorias"] / feat["n_licitacoes"]

# So fornecedores com atividade minima (senao vira ruido)
feat = feat[feat["n_licitacoes"] >= 5].copy()
print("Fornecedores analisados:", len(feat))

# --- Isolation Forest ---
cols = ["n_licitacoes", "n_vitorias", "n_ugs", "n_modalidades",
        "pct_dispensa", "pct_inexig", "taxa_vitoria"]
X = feat[cols].fillna(0)

iso = IsolationForest(contamination=0.02, random_state=42)
feat["anomaly_score"] = iso.fit_predict(X)
feat["score"] = iso.score_samples(X)  # quanto menor, mais anomalo

# --- Ranking dos mais anomalos ---
nomes = part.drop_duplicates("Código Participante").set_index("Código Participante")["Nome Participante"]
anomalos = feat.sort_values("score").head(20).copy()
anomalos["nome"] = anomalos.index.map(nomes)
anomalos[["nome", "n_licitacoes", "taxa_vitoria", "pct_dispensa", "pct_inexig", "n_ugs", "score"]]

Fornecedores analisados: 38641


,nome,n_licitacoes,taxa_vitoria,pct_dispensa,pct_inexig,n_ugs,score
Código Participante,,,,,,,
45769285000168,REDNOV FERRAMENTAS LTDA.,28130,0.282474,0.016317,0.000000,680,-0.779593
18707234000139,CLENEX COMERCIO E SERVICOS LTDA,15011,0.095397,0.000266,0.000000,250,-0.775800
-11,Sigiloso,15236,0.238383,0.102586,0.017524,35,-0.773289
35236131000157,GGV COMERCIAL LTDA,10688,0.423185,0.000000,0.000000,242,-0.771721
49834027000179,AGREGA DISTRIBUIDORA LTDA,11449,0.438641,0.001485,0.000000,237,-0.771721
21707794000106,FASTLABOR COMERCIAL LTDA,10712,0.492252,0.006441,0.000000,151,-0.771199
14968227000130,FERGAVI COMERCIAL LTDA,9146,0.216051,0.000219,0.000000,260,-0.770461
13338681000144,COMERCIAL SPONCHIADO LTDA,13429,0.274704,0.008638,0.000000,183,-0.769490
80243769000170,AMBARLAB PRODUTOS LABORATORIAIS LTDA,5134,0.558239,0.000390,0.000000,214,-0.768594
